# Direct Preference Optimization (DPO) — Bangla নোটবুক

এটি `example.py`-এর একটি Bangla-অনুবাদিত, চালানোযোগ্য নোটবুক সংস্করণ।

Direct Preference Optimization (DPO)

PyTorch-এ scratch থেকে DPO loss (Rafailov et al., 2023) implement করে
এবং এটি দিয়ে একটি ছোট decoder-only Transformer-কে (Phase 02 Lesson 6-এর
একই mini-GPT architecture; policy এবং frozen reference model -- দুইটিই
হিসেবে এখানে পুনরায় ব্যবহৃত) সরাসরি synthetic (prompt, chosen, rejected)
preference triple-তে fine-tune করে -- কখনো একটি আলাদা reward model
প্রশিক্ষণ না দিয়ে (Lesson 2) এবং কোনো RL rollout / PPO update ছাড়াই
(Lesson 3)। এটি অবিকল সেই সরলীকরণ যার জন্য DPO বিখ্যাত: একই Bradley-Terry
preference objective, একটি supervised-দেখতে loss এবং একটি একক backward
pass দিয়ে optimize করা।

এখানে খেলনা স্কেলে পাইপলাইনটি:
  1. কিছু prompt-এর POSITIVE-sentiment এবং NEGATIVE-sentiment -- দুই ধরনের
     ধারাবাহিকতাই অন্তর্ভুক্ত এমন সাধারণ টেক্সটে একটি ছোট LM-কে
     "pretrain" করুন (একটি SFT মডেলের পরিবর্ত যা দুই ধরনের ধারাবাহিকতা
     *দেখেছে* কিন্তু এখনও সেগুলোর মধ্যে কোনো পছন্দ করে না)।
  2. এর একটি কপি pi_ref হিসেবে freeze করুন।
  3. এটিকে আবার pi_theta হিসেবে ক্লোন করুন (যে policy-টি DPO আসলে update করবে)।
  4. preference triple-তে (README section 2) শুধুমাত্র DPO loss দিয়ে
     শুধু pi_theta-কে fine-tune করুন -- কোনো reward model নেই, কোনো
     sampling/rollout নেই, কোনো PPO নেই।
  5. প্রশিক্ষণ জুড়ে log-probability margin log pi(chosen) - log
     pi(rejected) বাড়তে দেখা -- প্রশিক্ষণ জোড়া এবং একটি ছোট held-out
     prompt/completion সেট, যেটির উপর DPO fine-tuning-এর সময় কখনো
     preference label দেওয়া হয়নি, দুইটির উপর (সততার সাথে যাচাই করতে
     যে শেখা পছন্দটি একটি generalization-যোগ্য "positive sentiment
     পছন্দ করো" নিয়ম নাকি নিছক মুখস্থকরণ)।

Runtime: CPU-তে এক মিনিটেরও ভালোভাবে কম।

চালান:
    নোটবুকের সব code cell উপরে থেকে নিচে চালান (Cell -> Run All)।

## কীভাবে চালাবেন

উপর থেকে নিচে (Cell → Run All) সব cell চালান। Runtime: CPU-তে এক মিনিটেরও কম।

In [ ]:
"""Direct Preference Optimization (DPO)

PyTorch-এ scratch থেকে DPO loss (Rafailov et al., 2023) implement করে
এবং এটি দিয়ে একটি ছোট decoder-only Transformer-কে (Phase 02 Lesson 6-এর
একই mini-GPT architecture; policy এবং frozen reference model -- দুইটিই
হিসেবে এখানে পুনরায় ব্যবহৃত) সরাসরি synthetic (prompt, chosen, rejected)
preference triple-তে fine-tune করে -- কখনো একটি আলাদা reward model
প্রশিক্ষণ না দিয়ে (Lesson 2) এবং কোনো RL rollout / PPO update ছাড়াই
(Lesson 3)। এটি অবিকল সেই সরলীকরণ যার জন্য DPO বিখ্যাত: একই Bradley-Terry
preference objective, একটি supervised-দেখতে loss এবং একটি একক backward
pass দিয়ে optimize করা।

এখানে খেলনা স্কেলে পাইপলাইনটি:
  1. কিছু prompt-এর POSITIVE-sentiment এবং NEGATIVE-sentiment -- দুই ধরনের
     ধারাবাহিকতাই অন্তর্ভুক্ত এমন সাধারণ টেক্সটে একটি ছোট LM-কে
     "pretrain" করুন (একটি SFT মডেলের পরিবর্ত যা দুই ধরনের ধারাবাহিকতা
     *দেখেছে* কিন্তু এখনও সেগুলোর মধ্যে কোনো পছন্দ করে না)।
  2. এর একটি কপি pi_ref হিসেবে freeze করুন।
  3. এটিকে আবার pi_theta হিসেবে ক্লোন করুন (যে policy-টি DPO আসলে update করবে)।
  4. preference triple-তে (README section 2) শুধুমাত্র DPO loss দিয়ে
     শুধু pi_theta-কে fine-tune করুন -- কোনো reward model নেই, কোনো
     sampling/rollout নেই, কোনো PPO নেই।
  5. প্রশিক্ষণ জুড়ে log-probability margin log pi(chosen) - log
     pi(rejected) বাড়তে দেখা -- প্রশিক্ষণ জোড়া এবং একটি ছোট held-out
     prompt/completion সেট, যেটির উপর DPO fine-tuning-এর সময় কখনো
     preference label দেওয়া হয়নি, দুইটির উপর (সততার সাথে যাচাই করতে
     যে শেখা পছন্দটি একটি generalization-যোগ্য "positive sentiment
     পছন্দ করো" নিয়ম নাকি নিছক মুখস্থকরণ)।

Runtime: CPU-তে এক মিনিটেরও ভালোভাবে কম।

চালান:
    নোটবুকের সব code cell উপরে থেকে নিচে চালান (Cell -> Run All)।
"""

import math
import re

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 0. খেলনা preference data: (prompt, chosen, rejected) triple

এখানে "preference" হলো একটি সহজ, পরিচিত ground-truth নিয়ম — chosen completion হলো positive-sentiment, rejected completion একই prompt-এর negative-sentiment ধারাবাহিকতা — যাতে পরে যাচাই করা যায় DPO training আসলে সেই নিয়মটি শিখেছে কিনা, নিছক ছয়টি sequence মুখস্থ করেনি।

In [ ]:
# ---------------------------------------------------------------------------
# 0. খেলনা preference data: (prompt, chosen, rejected) triple।
#
# এখানে "preference" হলো একটি সহজ, পরিচিত ground-truth নিয়ম -- chosen
# completion হলো positive-sentiment, rejected completion একই prompt-এর
# negative-sentiment ধারাবাহিকতা -- যাতে পরে যাচাই করা যায় DPO training
# আসলে সেই নিয়মটি শিখেছে, নিছক ছয়টি sequence মুখস্থ করেনি।
# ---------------------------------------------------------------------------

TRAIN_PAIRS = [
    ("the movie was", "absolutely wonderful and touching.", "absolutely dreadful and boring."),
    ("the food tasted", "incredibly delicious and fresh.", "incredibly bland and stale."),
    ("the service was", "truly excellent and attentive.", "truly rude and careless."),
    ("the weather today is", "bright sunny and pleasant.", "cold gloomy and miserable."),
    ("my whole day was", "wonderful and full of joy.", "terrible and full of stress."),
    ("the concert last night was", "amazing and unforgettable.", "awful and forgettable."),
]

# DPO fine-tuning থেকে সম্পূর্ণ বাদ দেওয়া হয়েছে -- শুধু generalization
# যাচাই করার জন্য ব্যবহৃত। এর WORDS নিচের pretraining কর্পাসে আছে (যাতে
# base model সেগুলো দেখেছে), কিন্তু DPO training কখনো তাদের উপর কোনো
# preference label দেখে না।
HELD_OUT_PAIRS = [
    ("the hotel room was", "clean bright and comfortable.", "dirty dark and uncomfortable."),
    ("the new phone is", "fast reliable and impressive.", "slow buggy and disappointing."),
]

ALL_PAIRS = TRAIN_PAIRS + HELD_OUT_PAIRS

## 1. একটি ছোট word-level tokenizer

শব্দ স্তরে log-probability নিয়ে ভাবা character-এর চেয়ে সহজ — কারণ preference signal-টি শব্দ স্তরে বাস করে ("wonderful" বনাম "dreadful")।

In [ ]:
# ---------------------------------------------------------------------------
# 1. একটি ছোট word-level tokenizer (শব্দ-স্তরের log-probability নিয়ে
# character-এর চেয়ে সহজে ভাবা যায় -- "wonderful" বনাম "dreadful" --
# এমন একটি preference signal-এর জন্য)।
# ---------------------------------------------------------------------------


def tokenize(text):
    return re.findall(r"[a-z]+|[.]", text.lower())


vocab = sorted({tok for prompt, chosen, rejected in ALL_PAIRS
                for tok in tokenize(prompt) + tokenize(chosen) + tokenize(rejected)})
PAD_ID = 0
stoi = {tok: i + 1 for i, tok in enumerate(vocab)}   # id 1..V, 0 PAD-এর জন্য সংরক্ষিত
itos = {i: tok for tok, i in stoi.items()}
VOCAB_SIZE = len(stoi) + 1


def encode(text):
    return [stoi[tok] for tok in tokenize(text)]

## 2. মডেল — Phase 02 Lesson 6-এর MiniGPT block নকশা

causal self-attention + feed-forward, Pre-LN residual block — শুধু এই খেলনা vocabulary-তে সেকেন্ডে প্রশিক্ষণের পক্ষে যথেষ্ট ছোট।

In [ ]:
# ---------------------------------------------------------------------------
# 2. মডেল: Phase 02 Lesson 6-এর MiniGPT-র সাথে অভিন্ন block নকশা
# (causal self-attention + feed-forward, Pre-LN residual block), শুধু
# এই খেলনা vocabulary-তে সেকেন্ডে প্রশিক্ষণের পক্ষে যথেষ্ট ছোট।
# ---------------------------------------------------------------------------

D_MODEL = 64
NUM_HEADS = 4
D_FF = 256
NUM_LAYERS = 2
BLOCK_SIZE = 20   # এই dataset-এর যেকোনো prompt+completion-এর চেয়ে স্বাচ্ছন্দ্যে লম্বা


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, block_size):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)).bool())

    def forward(self, x):
        batch, T, d_model = x.shape

        def split_heads(t):
            return t.view(batch, T, self.num_heads, self.d_k).transpose(1, 2)

        Q, K, V = split_heads(self.W_q(x)), split_heads(self.W_k(x)), split_heads(self.W_v(x))
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(~self.mask[:T, :T], float("-inf"))
        weights = F.softmax(scores, dim=-1)
        out = (weights @ V).transpose(1, 2).contiguous().view(batch, T, d_model)
        return self.W_o(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList(
            [DecoderBlock(d_model, num_heads, d_ff, block_size) for _ in range(num_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        batch, T = token_ids.shape
        positions = torch.arange(T, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        return self.output_head(x)   # (batch, T, vocab_size)

## 3. Padded (prompt, completion) tensor এবং completion-only log-probability

pretraining এবং DPO -- দুইটিরই যে যন্ত্রপাতি দরকার: "এই নির্দিষ্ট prompt দেওয়া থাকলে, মডেল এই নির্দিষ্ট completion token-গুলোকে কী log-probability দেয়?"


In [ ]:
# ---------------------------------------------------------------------------
# 3. Padded (prompt, completion) tensor এবং একটি completion-only
# log-probability function -- pretraining ও DPO -- দুইটিরই প্রয়োজনীয়
# যন্ত্রপাতি: "এই নির্দিষ্ট prompt দেওয়া থাকলে, মডেল এই নির্দিষ্ট
# completion token-গুলোকে কী log-probability দেয়?"
# ---------------------------------------------------------------------------


def build_example(prompt, completion):
    """ফেরত দেয় (full_token_ids, num_prompt_tokens)।"""
    prompt_ids = encode(prompt)
    completion_ids = encode(completion)
    return prompt_ids + completion_ids, len(prompt_ids)


def make_batch(pairs, which):
    """which {'chosen', 'rejected'}-এর মধ্যে। Padded (input, target, mask)
    tensor ফেরত দেয় আকারে (batch, BLOCK_SIZE - 1), যেখানে mask[i, t] = 1
    ঠিক সেই target পজিশনগুলোতে যা একটি COMPLETION token-এর সাথে মিলে
    (কখনো prompt token নয়, কখনো padding নয়)।"""
    inputs, targets, masks = [], [], []
    for prompt, chosen, rejected in pairs:
        completion = chosen if which == "chosen" else rejected
        full_ids, num_prompt = build_example(prompt, completion)
        n = len(full_ids)
        inp = full_ids[:-1] + [PAD_ID] * (BLOCK_SIZE - 1 - (n - 1))
        tgt = full_ids[1:] + [PAD_ID] * (BLOCK_SIZE - 1 - (n - 1))
        mask = [0] * (BLOCK_SIZE - 1)
        # target index i-তে full_ids[i+1] থাকে; এটি একটি completion token
        # যখন i+1 >= num_prompt, অর্থাৎ i >= num_prompt - 1।
        for i in range(num_prompt - 1, n - 1):
            mask[i] = 1
        inputs.append(inp)
        targets.append(tgt)
        masks.append(mask)
    return (torch.tensor(inputs, dtype=torch.long),
            torch.tensor(targets, dtype=torch.long),
            torch.tensor(masks, dtype=torch.float))


def sequence_logprobs(model, input_ids, target_ids, mask):
    """হুবহু masked (completion) পজিশন-গুলির উপর log p(target_t | input_<=t)
    যোগফল, batch-এর প্রতিটি sequence-এর জন্য -- এটি মডেলের নিজস্ব causal
    factorization-এর অধীনে log pi(completion | prompt)। আকৃতি (batch,) ফেরত দেয়।"""
    logits = model(input_ids)                                  # (batch, T, V)
    log_probs = F.log_softmax(logits, dim=-1)
    token_logprobs = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)  # (batch, T)
    return (token_logprobs * mask).sum(dim=1)                  # (batch,)

## 4. DPO loss নিজেই (README section 2)

কোথাও কোনো reward model নেই — `pi_ref` একই সাথে "reward baseline" এবং Lesson 3-এর KL anchor — দুইটিরই ভূমিকা পালন করে।

In [ ]:
# ---------------------------------------------------------------------------
# 4. DPO loss নিজেই (README section 2):
#
#   loss = -log( sigmoid( beta * [ (log pi(chosen|x) - log pi_ref(chosen|x))
#                                 - (log pi(rejected|x) - log pi_ref(rejected|x)) ] ) )
#
# কোথাও কোনো reward model নেই -- pi_ref একই সাথে "reward baseline" এবং
# Lesson 3-এর KL anchor -- দুইটিরই ভূমিকা একসাথে পালন করে।
# ---------------------------------------------------------------------------


def dpo_loss(policy, ref, chosen_batch, rejected_batch, beta):
    c_in, c_tgt, c_mask = chosen_batch
    r_in, r_tgt, r_mask = rejected_batch

    policy_chosen_logp = sequence_logprobs(policy, c_in, c_tgt, c_mask)
    policy_rejected_logp = sequence_logprobs(policy, r_in, r_tgt, r_mask)
    with torch.no_grad():
        ref_chosen_logp = sequence_logprobs(ref, c_in, c_tgt, c_mask)
        ref_rejected_logp = sequence_logprobs(ref, r_in, r_tgt, r_mask)

    pi_logratios = policy_chosen_logp - policy_rejected_logp
    ref_logratios = ref_chosen_logp - ref_rejected_logp
    implicit_reward_margin = pi_logratios - ref_logratios

    loss = -F.logsigmoid(beta * implicit_reward_margin).mean()
    return loss, policy_chosen_logp.detach(), policy_rejected_logp.detach()

## 5. Pretraining: একটি সাধারণ next-token cross-entropy LM

প্রতিটি prompt-এর chosen ও rejected উভয় ধারাবাহিকতাতেই (train + held-out একসাথে) trained — কোনো preference ধারণা ছাড়াই। একটি SFT মডেল যা ঠিক তেমন: সাবলীল, এবং কোনো preference-ভিত্তিক stage এটিকে স্পর্শ করার আগে দুই ধারাবাহিকতাই তৈরি করতে সক্ষম।

In [ ]:
# ---------------------------------------------------------------------------
# 5. Pretraining: একটি সাধারণ next-token cross-entropy LM, প্রতিটি
# prompt-এর chosen এবং rejected -- দুই ধরনের ধারাবাহিকতাতেই (train +
# held-out একইভাবে) প্রশিক্ষিত, কোনো preference ধারণা ছাড়াই -- একটি
# SFT মডেল ঠিক যা: সাবলীল, এবং কোনো preference-ভিত্তিক stage স্পর্শ
# করার আগে দুই ধারাবাহিকতাই তৈরি করতে সক্ষম।
# ---------------------------------------------------------------------------


def pretrain(model, pairs, steps=300, lr=3e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    chosen_batch = make_batch(pairs, "chosen")
    rejected_batch = make_batch(pairs, "rejected")
    for step in range(1, steps + 1):
        optimizer.zero_grad()
        total_loss = 0.0
        for (inp, tgt, mask) in (chosen_batch, rejected_batch):
            logits = model(inp)
            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1), ignore_index=PAD_ID
            )
            loss.backward()
            total_loss += loss.item()
        optimizer.step()
        if step % 100 == 0 or step == 1:
            print(f"  pretrain step {step:4d}   LM cross-entropy loss = {total_loss / 2:.4f}")


def average_margin(model, pairs):
    chosen_batch = make_batch(pairs, "chosen")
    rejected_batch = make_batch(pairs, "rejected")
    with torch.no_grad():
        c_logp = sequence_logprobs(model, *chosen_batch)
        r_logp = sequence_logprobs(model, *rejected_batch)
    margin = (c_logp - r_logp)
    accuracy = (margin > 0).float().mean().item()
    return margin.mean().item(), accuracy

## সম্পূর্ণ ডেমো চালানো

`main()`: pretraining → frozen `pi_ref` + `pi_theta` ক্লোন → শুধুমাত্র DPO loss দিয়ে fine-tuning → chosen-বনাম-rejected log-probability margin আসলে বেড়েছে কিনা (train ও held-out দুই সেটেই) যাচাই।

In [ ]:
def main():
    print("=" * 70)
    print("0. SETUP")
    print("=" * 70)
    print(f"Vocabulary size (word-level, toy corpus): {VOCAB_SIZE}")
    print(f"{len(TRAIN_PAIRS)} training preference triples, "
          f"{len(HELD_OUT_PAIRS)} held-out triples (no preference label used in DPO training)")
    print("Ground-truth rule the preference data encodes: chosen = positive-sentiment")
    print("continuation, rejected = negative-sentiment continuation of the same prompt.")

    print("\n" + "=" * 70)
    print("1. PRETRAINING A BASE (SFT-LIKE) MODEL ON BOTH CHOSEN AND REJECTED TEXT")
    print("=" * 70)
    print("This model sees BOTH sentiments as plain, equally-plausible text -- it has")
    print("no preference yet, exactly like an SFT model before any alignment stage.")
    base_model = MiniGPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, BLOCK_SIZE)
    pretrain(base_model, ALL_PAIRS, steps=300)

    ref_model = MiniGPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, BLOCK_SIZE)
    ref_model.load_state_dict(base_model.state_dict())
    for p in ref_model.parameters():
        p.requires_grad_(False)
    ref_model.eval()   # pi_ref: এখান থেকে frozen -- Lesson 3-র KL anchor-এর মতোই

    policy_model = MiniGPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, BLOCK_SIZE)
    policy_model.load_state_dict(base_model.state_dict())   # pi_theta pi_ref-এর সাথে অভিন্ন অবস্থা থেকে শুরু

    train_margin_before, train_acc_before = average_margin(policy_model, TRAIN_PAIRS)
    heldout_margin_before, heldout_acc_before = average_margin(policy_model, HELD_OUT_PAIRS)
    print(f"\nBEFORE any DPO training (policy == reference model):")
    print(f"  avg [log pi(chosen) - log pi(rejected)] on TRAIN pairs   = {train_margin_before:+.3f}"
          f"   (chosen preferred in {train_acc_before * 100:.0f}% of pairs)")
    print(f"  avg [log pi(chosen) - log pi(rejected)] on HELD-OUT pairs = {heldout_margin_before:+.3f}"
          f"   (chosen preferred in {heldout_acc_before * 100:.0f}% of pairs)")
    print("-> Close to a coin flip on both -- the pretrained model has no systematic")
    print("   preference for positive over negative sentiment; it just learned both are fluent.")

    print("\n" + "=" * 70)
    print("2. DPO FINE-TUNING -- NO REWARD MODEL, NO RL, JUST THIS LOSS:")
    print("=" * 70)
    print("loss = -log( sigmoid( beta * [ (log pi(c) - log pi_ref(c)) - (log pi(r) - log pi_ref(r)) ] ) )")
    print("Optimized directly on the 6 TRAINING preference pairs. pi_ref is frozen throughout.\n")

    BETA = 0.5
    LR = 5e-4
    NUM_STEPS = 400
    optimizer = torch.optim.Adam(policy_model.parameters(), lr=LR)

    chosen_batch = make_batch(TRAIN_PAIRS, "chosen")
    rejected_batch = make_batch(TRAIN_PAIRS, "rejected")

    history = []
    for step in range(1, NUM_STEPS + 1):
        optimizer.zero_grad()
        loss, c_logp, r_logp = dpo_loss(policy_model, ref_model, chosen_batch, rejected_batch, BETA)
        loss.backward()
        optimizer.step()
        margin = (c_logp - r_logp).mean().item()
        history.append((step, loss.item(), margin))
        if step % 50 == 0 or step == 1:
            print(f"  DPO step {step:4d}   loss = {loss.item():.4f}   "
                  f"avg train margin log pi(chosen)-log pi(rejected) = {margin:+.3f}")

    print("\n" + "=" * 70)
    print("3. RESULTS: DID THE LOG-PROBABILITY MARGIN ACTUALLY GROW?")
    print("=" * 70)
    train_margin_after, train_acc_after = average_margin(policy_model, TRAIN_PAIRS)
    heldout_margin_after, heldout_acc_after = average_margin(policy_model, HELD_OUT_PAIRS)

    print(f"{'':>28}{'before DPO':>14}{'after DPO':>14}")
    print(f"{'TRAIN margin':>28}{train_margin_before:>14.3f}{train_margin_after:>14.3f}")
    print(f"{'TRAIN pairwise accuracy':>28}{train_acc_before:>14.2f}{train_acc_after:>14.2f}")
    print(f"{'HELD-OUT margin':>28}{heldout_margin_before:>14.3f}{heldout_margin_after:>14.3f}")
    print(f"{'HELD-OUT pairwise accuracy':>28}{heldout_acc_before:>14.2f}{heldout_acc_after:>14.2f}")

    print(f"\n-> On the {len(TRAIN_PAIRS)} TRAINING pairs, the margin log pi(chosen) - log pi(rejected)")
    print(f"   moved from {train_margin_before:+.3f} to {train_margin_after:+.3f} purely by minimizing the DPO loss --")
    print(f"   no reward model was ever instantiated, and no text was ever sampled/rolled")
    print(f"   out from the policy during this entire training loop.")

    if heldout_acc_after > heldout_acc_before:
        print(f"\n-> On the {len(HELD_OUT_PAIRS)} HELD-OUT pairs (no preference label used during DPO training,")
        print(f"   only seen as plain text during pretraining), pairwise accuracy also rose, from")
        print(f"   {heldout_acc_before:.2f} to {heldout_acc_after:.2f} -- evidence the model shifted probability mass")
        print(f"   toward positive-sentiment completions in general, not merely toward the six")
        print(f"   exact training sequences it was fine-tuned on.")
    else:
        print(f"\n-> On the {len(HELD_OUT_PAIRS)} HELD-OUT pairs, pairwise accuracy went from {heldout_acc_before:.2f} to")
        print(f"   {heldout_acc_after:.2f}. With only {len(TRAIN_PAIRS)} training pairs and a model this tiny, DPO here")
        print(f"   mainly memorizes the specific training sequences rather than learning a fully")
        print(f"   general 'prefer positive sentiment' direction -- a real system needs far more")
        print(f"   preference data and a far larger backbone for the implicit reward to generalize.")

    print(f"\nSample: implicit rewards beta*(log pi/pi_ref) for one TRAIN pair before vs after DPO")
    print(f"(this log-ratio is exactly the reward DPO optimizes WITHOUT ever training an explicit")
    print(f"reward model -- README section 3):")
    prompt, chosen, rejected = TRAIN_PAIRS[0]
    one_chosen = make_batch([TRAIN_PAIRS[0]], "chosen")
    one_rejected = make_batch([TRAIN_PAIRS[0]], "rejected")
    with torch.no_grad():
        ref_c = sequence_logprobs(ref_model, *one_chosen).item()
        ref_r = sequence_logprobs(ref_model, *one_rejected).item()
        pol_c = sequence_logprobs(policy_model, *one_chosen).item()
        pol_r = sequence_logprobs(policy_model, *one_rejected).item()
    print(f"  prompt = {prompt!r}")
    print(f"  chosen = {chosen!r}  |  rejected = {rejected!r}")
    print(f"  implicit reward of chosen   = beta*(log pi - log pi_ref) = "
          f"{BETA * (pol_c - ref_c):+.3f}")
    print(f"  implicit reward of rejected = beta*(log pi - log pi_ref) = "
          f"{BETA * (pol_r - ref_r):+.3f}")
    print("  -> DPO raised the implicit reward of the chosen response relative to the")
    print("     reference and lowered it for the rejected response, using only supervised-style")
    print("     gradient descent on labeled pairs -- exactly the RLHF objective from Lesson 2/3,")
    print("     reached without a reward model or an RL rollout loop.")

main()